# make database

In [20]:
import pandas as pd
import numpy as np


## mostly from all_consonants_data
- does not include vowels (46-52)
- does not include reference words/sound changes

In [21]:
all_consonants = pd.read_csv('../data_processing/all_consonants_data.csv')

In [22]:
all_consonants.rename(columns={'IPA key': 'IPA_key',  'Consonantal +/−': 'consonantal', 'Voice +/−': 'voice', 'Sibilant +/−': 'sibilant', 'Lateral +/−': 'lateral',
                               'Place of articulation': 'place_articulation', 'Place value': 'place_value', 'Manner of articulation': 'articulation_manner', 'Sonority value': 'consonantality_value'}, 
                     inplace=True)



### ConsonantIPA

- identify inconsistencies in original data (mislabeled IPA keys, unexpected data)

In [ ]:
# get all unique IPA/data combinations
original_IPAs = all_consonants[['IPA_key', 'IPA', 'consonantal', 'voice', 'sibilant', 'lateral', 'place_articulation', 'place_value', 'articulation_manner', 'consonantality_value']].drop_duplicates()

# remove phi keys for now
drop_keys = ['∅', None]
original_IPAs = original_IPAs[~original_IPAs['IPA_key'].isin(drop_keys)].dropna(subset = ['IPA_key'])

original_IPAs['IPA_key'] = original_IPAs['IPA_key'].astype(float)
original_IPAs.sort_values(by= ['IPA_key'], inplace=True)

original_IPAs.to_csv('original_IPAs.csv')


## compare to IPA in excel doc
    # where are there duplicates in the original full excel
    # and how many are different from IPA.csv


# duplicates in excel after finding all unique sets of data with each IPA
excel_IPAs = original_IPAs.groupby('IPA').size().reset_index(name = 'count')
excel_IPAs.sort_values('count', inplace=True)
excel_dupes = excel_IPAs[excel_IPAs['count'] > 1]

# get original IPAs where multiple versions occured
dupe_IPAs = original_IPAs[original_IPAs['IPA'].isin(excel_dupes['IPA'])].sort_values('IPA')

dupe_IPAs.to_csv('duplicate_data.csv')
    # cleaned instances that were obviously mislabeled, subset left have very small differences, will assign data based off of IPA key and new table



- then check mapping ability

In [75]:
# read in full set of new IPA key data
new_IPAs = pd.read_csv('newConsonantIPA.csv', usecols = range(11)).dropna(subset = ['key']) # remove unusued columns and rows

new_IPAs.rename(columns = {'key': 'IPA_key', 'Consonantal?': 'consonantal', 'Voiced?': 'voice', 'Sibilant?': 'sibilant', 'Lateral?': 'lateral', 'Geminate?': 'geminate',
                            'Place of\narticulation': 'place_articulation', 'Place\nvalue': 'place_value', 'Manner of\narticulation': 'articulation_manner', 'Consonantality\nvalue': 'consonantality_value'},
                inplace = True)



# get original IPA keys
og_consonant_keys = original_IPAs[['IPA', 'IPA_key']].drop_duplicates()

og_consonant_keys['IPA'].astype(str)
new_IPAs['IPA'].astype(str)

#merge old with new keys on IPA to check existence from old:new as they appear in the data
joined_keys = og_consonant_keys.merge(new_IPAs[['IPA_key', 'IPA']], how = 'right', on = 'IPA')


# check that manual mapping matches old_to_new_keys.csv
old_to_new_keys = pd.read_csv('old_to_new_keys.csv')
    # going to use old_to_new_keys as the official mapping, since not all original IPA keys show up in the data (and therefore aren't in joined_keys)

keys_test = joined_keys.merge(old_to_new_keys, how = 'right', on = 'IPA')
#keys_test.to_csv('keys_test.csv')
    # confirmed, manual mapping matches official mapping where data is present


- then add relabeled IPA keys to consonantIPA

In [77]:
consonantIPA = new_IPAs.merge(old_to_new_keys, how = 'left', right_on = 'new key', left_on = 'IPA_key')


consonantIPA = consonantIPA.iloc[:, [0, 1, 11, 2, 3, 4, 5, 6, 7, 8, 9, 10]]
consonantIPA.rename({'IPA_x': 'IPA', 'old key': 'old_key'}, inplace = True)

consonantIPA.to_csv('tables/consonantIPA.csv')

### TrtmtEnvSegments

In [ ]:
trtmtEnvSegments = all_consonants[['number', 'Languages', 'position', 'version', 'IPA', 'IPA key']].drop_duplicates()

trtmtEnvSegments.rename(columns = {'number': 'trtmt_envID', 'Languages': 'language', 'IPA key': 'IPA_key'}, inplace = True)

trtmtEnvSegments['is_vowel'] = False

In [ ]:
trtmtEnvSegments.head(10)

,trtmt_envID,language,position,version,IPA,IPA_key,is_vowel
0,1.0,BR POR,1,a,b,2,True
1,1.0,AMR ESP,1,a,b,2,True
2,1.0,LAT,1,a,b,2,True
3,1.0,CAT,1,a,b,2,True
4,1.0,VAL,1,a,b,2,True
5,1.0,BAL,1,a,b,2,True
6,1.0,OCC,1,a,b,2,True
7,1.0,GAS OCC,1,a,b,2,True
8,1.0,EU FRA,1,a,b,2,True
9,1.0,QBC FRA,1,a,b,2,True


### TrtmtEnv

In [ ]:
trtmtEnv = all_consonants[['number', 'treatment', 'environment']].drop_duplicates()

trtmtEnv.rename(columns = {'number': 'trtmt_envID'}, inplace = True)

In [ ]:
print(len(trtmtEnv))
print(trtmtEnv.head(10))

100
    trtmt_envID treatment environment
0           1.0        B-     #_{a,O}
33          2.0        B-         #_E
66          3.0        T-         #_E
103         4.0        T-     #_{a,O}
137         5.0        D-         #_E
174         6.0        D-     #_{a,O}
208         7.0        C-         #_E
237         8.0        C-         #_a
270         9.0        C-     #_{o,ŭ}
304        10.0        C-         #_ū
